# Colab Ollama Runtime - Qwen 3 for the Analytics Engine

Run Ollama with `qwen3:8b` on Colab's free T4 GPU, then point your local machine at it.

## Workflow

1. Run all cells in this notebook (first run downloads Ollama + the model, so it takes a few minutes).
2. The last output prints `OLLAMA_BASE_URL` and `OLLAMA_API_KEY`.
3. On your laptop:
   - switch to Colab: `python switch_llm.py colab <OLLAMA_BASE_URL> <OLLAMA_API_KEY>`
   - run as usual, e.g. `python run_sample.py dbms_analytics_test`
   - switch back: `python switch_llm.py local`
   - check the current backend: `python switch_llm.py status`

> The tunnel URL is public; the `OLLAMA_API_KEY` protects it. Keep it secret.


## Settings

Edit `REPO_URL` below (a fork or private repo works too - set a `GITHUB_TOKEN`
secret in Colab -> Secrets if it is private). Set `FULL_APP = True` to also run
the FastAPI dashboard on Colab.


In [ ]:
import os
import re
import secrets
import subprocess
import time

REPO_URL = "https://github.com/R26-SE-025/V2_QuestionExamPredictionEngine.git"
FULL_APP = False  # set True to also run the FastAPI dashboard on Colab

API_KEY = secrets.token_hex(16)
print(f"OLLAMA_API_KEY={API_KEY}")


In [ ]:
repo = os.path.basename(REPO_URL).replace(".git", "")
!rm -rf repo
if os.environ.get("GITHUB_TOKEN"):
    REPO_URL = REPO_URL.replace(
        "https://github.com/", f"https://{os.environ['GITHUB_TOKEN']}@github.com/"
    )
!git clone {REPO_URL} repo
os.chdir("repo")
print("cloned into", repo)


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_API_KEY": API_KEY}
print("starting ollama serve with API key")
subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        subprocess.run(
            ["curl", "-s", "-f", "http://localhost:11434/api/tags"],
            capture_output=True,
            timeout=5,
        )
        break
    except Exception:
        time.sleep(2)
print("ollama up")


In [ ]:
!ollama pull qwen3:8b


In [ ]:
!curl -L -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared


In [ ]:
log = open("/tmp/cf.log", "w")
subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=log,
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    text = open("/tmp/cf.log").read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break
    time.sleep(2)
assert url, "tunnel URL not found in /tmp/cf.log"
print(f"OLLAMA_BASE_URL={url}")
print(f"OLLAMA_API_KEY={API_KEY}")


## On your laptop

Run this (replace the URL and key with the printed values):

```
python switch_llm.py colab https://<id>.trycloudflare.com <api-key>
python run_sample.py dbms_analytics_test
```

Switch back to local Ollama anytime:

```
python switch_llm.py local
```


In [ ]:
if FULL_APP:
    from google.colab import userdata

    try:
        mongo_uri = userdata.get("MONGODB_URI")
    except Exception:
        mongo_uri = input("Paste MONGODB_URI: ").strip()
    os.environ["MONGODB_URI"] = mongo_uri
    os.environ["MONGODB_DB"] = "Grading"
    os.environ["OLLAMA_API_KEY"] = API_KEY
    print("env configured for full app")


In [ ]:
if FULL_APP:
    !pip install -q -r requirements.txt


In [ ]:
if FULL_APP:
    log = open("/tmp/uvicorn.log", "w")
    subprocess.Popen(
        ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    log2 = open("/tmp/cf2.log", "w")
    subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=log2,
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(60):
        text = open("/tmp/cf2.log").read()
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if match:
            url = match.group(0)
            break
        time.sleep(2)
    print(f"DASHBOARD_URL={url}")
